In [ ]:
# Cell 1: Imports
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, WebDriverException

from bs4 import BeautifulSoup
import time
from datetime import datetime
import re
import random
from urllib.parse import urljoin

import undetected_chromedriver as uc

import psycopg2
from psycopg2 import sql
from dotenv import load_dotenv
import os

print("Cell 1: Imports đã sẵn sàng.")

In [ ]:
# Cell 2: Configuration
# --- Tải biến môi trường từ file .env ---
load_dotenv()

# --- Các hằng số cho scraping ---
TARGET_URL_BASE = "https://www.topcv.vn"
INITIAL_TARGET_URL = "https://www.topcv.vn/tim-viec-lam-moi-nhat-tai-ho-chi-minh-l2"
MAX_PAGES_TO_SCRAPE = 15 # số trang mục tiêu, nhưng script sẽ tự dừng nếu hết trang.

# --- Cấu hình Database ---
DB_HOST = os.getenv('DB_HOST')
DB_NAME = os.getenv('DB_NAME')
DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_PORT = os.getenv('DB_PORT', '5432')

# Tên bảng mới để lưu hàng đợi URL. Rất quan trọng!
DB_URL_QUEUE_TABLE = 'topcv_job_url_queue'

print("Cell 2: Cấu hình đã được thiết lập.")
print(f"Bảng lưu URL: {DB_URL_QUEUE_TABLE}")

In [ ]:
# Cell 3: WebDriver Initialization
driver = None
try:
    print("Cell 3: Đang cấu hình và khởi tạo WebDriver...")
    chrome_options_uc = Options()
    
    # User-agent của cậu
    USER_AGENT = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36"
    chrome_options_uc.add_argument(f"--user-agent={USER_AGENT}")
    chrome_options_uc.add_argument("--window-size=1920,1080")
    # KHÔNG TẮT ẢNH
    
    # Giúp tránh bị phát hiện là bot
    chrome_options_uc.add_argument('--disable-blink-features=AutomationControlled')
    
    driver = uc.Chrome(options=chrome_options_uc)
    
    # Tăng thời gian chờ tải trang mặc định của Selenium lên 3 phút
    driver.set_page_load_timeout(180) 
    
    print("WebDriver đã được khởi tạo và cấu hình thành công!")

except WebDriverException as e:
    print(f"Lỗi nghiêm trọng khi khởi tạo WebDriver: {e}")
    print("Hãy kiểm tra phiên bản ChromeDriver và trình duyệt Chrome có tương thích không.")
    driver = None
except Exception as e:
    print(f"Lỗi không xác định khi khởi tạo WebDriver: {e}")
    driver = None

In [ ]:
# Cell 4: Main URL Scraping Loop (PHIÊN BẢN SỬA LỖI VÀ ỔN ĐỊNH)

# Kiểm tra xem driver và các biến DB đã sẵn sàng chưa
if driver and all([DB_HOST, DB_NAME, DB_USER, DB_PASSWORD]):
    conn = None
    cur = None
    urls_added_this_session = 0
    
    try:
        # 1. Kết nối đến kho (Database)
        print("Đang kết nối đến PostgreSQL...")
        conn = psycopg2.connect(dbname=DB_NAME, user=DB_USER, password=DB_PASSWORD, host=DB_HOST, port=DB_PORT)
        cur = conn.cursor()
        print("Kết nối thành công!")

        # 2. Bắt đầu hành trình
        current_page_num = 1
        
        while current_page_num <= MAX_PAGES_TO_SCRAPE:
            
            # 3. TỰ XÂY DỰNG URL CHO TRANG HIỆN TẠI - ĐÂY LÀ THAY ĐỔI QUAN TRỌNG
            # Bằng cách này, chúng ta không phụ thuộc vào nút "Next"
            url_to_scrape = f"{INITIAL_TARGET_URL}?page={current_page_num}"
            
            print(f"\\n--- Đang khảo sát trang {current_page_num} ---")
            print(f"URL: {url_to_scrape}")
            
            try:
                # 4. Đi đến địa điểm (URL)
                driver.get(url_to_scrape)

                # 5. Chờ cho đến khi thấy "khu dân cư" (danh sách job)
                WebDriverWait(driver, 30).until(
                    EC.presence_of_element_located((By.CLASS_NAME, "box-job-list"))
                )
                print("  Đã thấy danh sách jobs. Đợi thêm chút để trang ổn định...")
                time.sleep(random.uniform(3, 5))

                # 6. Chụp lại bản đồ của khu vực (lấy HTML)
                soup = BeautifulSoup(driver.page_source, 'html.parser')

                # 7. Tìm tất cả các "ngôi nhà" (job item) trên bản đồ
                job_links = soup.select('div.job-item-search-result div.title-block h3 a')
                print(f"  Tìm thấy {len(job_links)} link trên trang này.")
                
                # *** ĐIỀU KIỆN DỪNG QUAN TRỌNG ***
                # Nếu trang không có link nào, tức là đã hết job, ta nên dừng lại.
                if not job_links:
                    print("  Không tìm thấy job nào trên trang này. Có thể đã đến trang cuối cùng. Kết thúc khảo sát.")
                    break

                page_urls_added = 0
                for link in job_links:
                    href = link.get('href')
                    if href:
                        # 8. Ghi lại địa chỉ của từng ngôi nhà
                        full_job_url = urljoin(TARGET_URL_BASE, href)
                        clean_url = full_job_url.split('?')[0]

                        # 9. Cất địa chỉ vào kho
                        insert_query = sql.SQL(
                            "INSERT INTO {} (job_url) VALUES (%s) ON CONFLICT (job_url) DO NOTHING;"
                        ).format(sql.Identifier(DB_URL_QUEUE_TABLE))
                        
                        cur.execute(insert_query, (clean_url,))
                        
                        if cur.rowcount > 0:
                            page_urls_added += 1
                
                urls_added_this_session += page_urls_added
                print(f"  Đã thêm mới {page_urls_added} URL vào hàng đợi.")
                conn.commit()

                # 10. Chuyển sang trang tiếp theo chỉ bằng cách tăng số trang
                current_page_num += 1
                
                # 11. Nghỉ chân trước khi đi tiếp
                print(f"  Nghỉ {random.uniform(5, 10):.1f} giây...")
                time.sleep(random.uniform(5, 10))

            except TimeoutException:
                print(f"  Hết thời gian chờ khi tải trang {current_page_num}. Có thể đây là trang không hợp lệ hoặc mạng chậm. Kết thúc.")
                break # Nếu timeout, có thể là do page không tồn tại, nên dừng lại
            except Exception as e:
                print(f"  Gặp lỗi không mong muốn ở trang {current_page_num}: {e}")
                break

    except psycopg2.Error as db_err:
        print(f"Lỗi Database: {db_err}")
    except Exception as general_err:
        print(f"Lỗi chung của script: {general_err}")
    finally:
        # 12. Kết thúc công việc, dọn dẹp
        print(f"\\n--- KẾT THÚC SCRIPT 1 ---")
        print(f"Tổng số trang đã xử lý: {current_page_num - 1}")
        print(f"Tổng số URL mới đã được thêm vào DB trong phiên này: {urls_added_this_session}")
        if cur: cur.close()
        if conn: conn.close(); print("Đã đóng kết nối DB.")
        if driver: driver.quit(); print("Đã đóng WebDriver.")
        
else:
    print("Script không thể chạy do WebDriver hoặc cấu hình DB bị thiếu.")